In [ ]:
# packages
import os
import joblib
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import GradientBoostingRegressor as GBR
from sklearn.model_selection import GridSearchCV

# set seed
seed = 5151

### You'll need the California housing data (CAhousing.csv) to complete this exercise. Your goal is to predict the median house value.

In [ ]:
CAhousing = pd.read_csv('data/CAhousing.csv')
CAhousing.head()

In [ ]:
CAhousing.dtypes

In [ ]:
CAhousing.shape

### There is a single categorical variable in this dataset that we will convert to a series of binary variables.

In [ ]:
CAhousing['ocean_proximity'].value_counts()

In [ ]:
# create binary variables for levels which have enough records, using '<1H OCEAN' as base.
CAhousing['ocean_prox_inland'] = (CAhousing.ocean_proximity == 'INLAND').astype(int)
CAhousing['ocean_prox_nearocean'] = (CAhousing.ocean_proximity == 'NEAR OCEAN').astype(int)
CAhousing['ocean_prox_nearbay'] = (CAhousing.ocean_proximity == 'NEAR BAY').astype(int)

# check for intended outcome
print(CAhousing['ocean_prox_inland'].value_counts())
print(CAhousing['ocean_prox_nearocean'].value_counts())
print(CAhousing['ocean_prox_nearbay'].value_counts())

In [ ]:
indep_vars = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 
              'total_bedrooms', 'population', 'households', 'median_income', 
              'ocean_prox_inland', 'ocean_prox_nearocean', 'ocean_prox_nearbay']

X = CAhousing[indep_vars]
y = CAhousing['median_house_value']

In [ ]:
X_train, X_test, y_train, y_test, Train, Test = train_test_split(X, y, CAhousing, 
                                                                 random_state = seed, 
                                                                 test_size = 0.33, 
                                                                 shuffle = True)

In [ ]:
X_train.corr().style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1)

In [ ]:
Train.describe()

### We see that `total_bedrooms` has a few missing values, and it is also very correlated with `total_rooms`. Because of this, it's easiest to remove it from our training set.

In [ ]:
indep_vars.remove("total_bedrooms")

### We'll define some helper functions here: 
(1) to calculate MSE, and 
(2) to display the improvement of a GBM on test data as additional trees are constructed.

In [ ]:
def mse(y, y_hat):
    """
    Calculate the MSE of a model given actual and predicted values
    """
    resid = y - y_hat
    sq_resid = resid**2
    SSR = sum(sq_resid)
    MSE = SSR / y.shape[0]
    return MSE

def gbm_progress(model_name, X_test, y_test):
    """
    Plot the 'progress' of a GBM on a test set to assess robustness
    """
    test_error = np.zeros_like(model_name.train_score_)
    for idx, y_ in enumerate(model_name.staged_predict(X_test)):
        test_error[idx] = np.mean((y_test - y_)**2)

    plot_idx = np.arange(model_name.train_score_.shape[0])
    ax = subplots(figsize=(8,8))[1]
    ax.plot(plot_idx, model_name.train_score_, 'b', label='Training')
    ax.plot(plot_idx, test_error, 'r', label='Test')
    ax.legend();

### Build a basic GBM first.

In [ ]:
gbm_basic = GBR(learning_rate=0.1,
               n_estimators=250,
               max_depth=10,
               min_samples_split=2,
               subsample=1,
               random_state=seed)

gbm_basic.fit(X_train[indep_vars], y_train)

In [ ]:
y_hat_basic = gbm_basic.predict(X_test[indep_vars])
mse_basic = mse(y_test, y_hat_basic)
print('test mse: ', mse_basic)

In [ ]:
gbm_progress(gbm_basic, X_test[indep_vars], y_test)

### Now you'll have the chance to choose optimal hyperparameters to build a better GBM.
Once the candidate hyperparameters are chosen, use 5-fold cross-validation to decide which collection of hyperparameters are best.

**v3 strategy:** V2 best was `lr=0.05, max_depth=7, min_samples_leaf=10, n_estimators=500, subsample=0.8`.

Key trends from v1 → v2: lower lr won, more trees won, stochastic boosting (subsample < 1) won.
All three hit their ceiling in v2, so v3 continues pushing in the same direction:
- `n_estimators` up to 700 and 1000 — 500 was the max and won, likely still not plateaued
- `learning_rate` down to 0.01 and 0.03 — pairs well with more trees
- `subsample` down to 0.6 and 0.7 — 0.8 won over 1.0, so more randomness may still help
- `max_depth` fine-tuned to 6, 7, 8 — tighter bracket around the winner
- `min_samples_leaf` fine-tuned to 8, 10, 12 — tighter bracket around the winner

Total combinations: 2 x 3 x 3 x 3 x 3 = 162, x5 folds = 810 fits

In [ ]:
def get_best_gbm(X, y, results_path="gridsearch_results.pkl", force_retrain=False):
    """
    Runs GridSearchCV OR loads precomputed results if available.

    Parameters:
        X, y : training data
        results_path : where results are saved
        force_retrain : if True, ignore saved file and rerun

    Returns:
        best_model : fitted model
        best_params : dict
    """

    # Fast Version: load existing results
    if os.path.exists(results_path) and not force_retrain:
        data = joblib.load(results_path)
        print("Loaded precomputed GridSearchCV results.")
        return data["best_model"], data["best_params"]

    # Slow Version: run grid search via cross-validation
    print("Running GridSearchCV...")

    gbm_cv = GBR(random_state=seed)

    # v3 grid: continuing to push in the direction v2 trends pointed
    # v2 best: lr=0.05, max_depth=7, min_samples_leaf=10, n_estimators=500, subsample=0.8
    param_grid = {
        "n_estimators":      [700, 1000],        # 500 was ceiling and won; push further
        "learning_rate":     [0.01, 0.03, 0.05], # go lower; pairs with more trees
        "max_depth":         [6, 7, 8],          # tight bracket around winner (7)
        "min_samples_leaf":  [8, 10, 12],        # tight bracket around winner (10)
        "subsample":         [0.6, 0.7, 0.8]     # 0.8 won over 1.0; try more randomness
    }

    grid_search = GridSearchCV(
        estimator=gbm_cv,
        param_grid=param_grid,
        cv=5,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        verbose=3
    )

    grid_search.fit(X, y)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    joblib.dump({
        "best_model": best_model,
        "best_params": best_params,
        "cv_results": grid_search.cv_results_
    }, results_path)

    print(f"Saved results to {results_path}")

    return best_model, best_params

In [ ]:
# NOTE: force_retrain=True so it ignores any old saved .pkl and runs fresh
best_model, best_params = get_best_gbm(X_train[indep_vars], y_train, force_retrain=True)

print("Best parameters:")
print(best_params)

### Your grade on this assignment will be partially determined by the mean squared error on a separate holdout set. You can approximate this using the test set.

In [ ]:
# Evaluate on test set
y_pred = best_model.predict(X_test[indep_vars])
mse_best = mse(y_test, y_pred)

print("Test MSE:", mse_best)